# Etapa 14 · Ampliación del corpus histórico de Memoria Chilena

**No repitas las etapas 1–13.** Este notebook parte del catálogo que ya está incluido en `data/output/etapa7/` y explora tres minisitios adicionales. Genera automáticamente `data/output/etapa14.zip`.

**Qué hace:** recopila hasta 8 fichas por minisitio, descarga HTML y RDF catalográficos, conserva títulos, autores, unidades, derechos y procedencia, y construye un grafo de co-pertenencia *curatorial*.

**Qué no hace:** no descarga PDF, no ejecuta OCR, no envía información a Groq y no genera afirmaciones históricas mediante LLM.

Si ejecutas este notebook nuevamente, reutiliza los HTML/RDF guardados en `data/cache/etapa14/`. Respeta `robots.txt` y su pausa. Los resultados contienen errores explícitos si alguna ficha no está accesible. No necesitas mover ZIP ni cambiar rutas.

In [1]:
%pip -q install requests beautifulsoup4 rdflib pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import csv, io, json, re, hashlib, time, zipfile, unicodedata
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests
from bs4 import BeautifulSoup, Tag
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from rdflib.namespace import DC, DCTERMS, PROV
from IPython.display import display, FileLink

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT/'notebooks').is_dir() and (ROOT/'data').is_dir(), (
    f'Abre este notebook desde la carpeta notebooks/ del proyecto. Carpeta actual: {Path.cwd()}'
)
OUT = ROOT/'data'/'output'
CACHE = ROOT/'data'/'cache'/'etapa14'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
BASE_CSV = OUT/'etapa7'/'catalogo_para_busqueda.csv'
if not BASE_CSV.is_file():
    BASE_CSV = ROOT/'data'/'reference'/'catalogo_base_etapa7.csv'
assert BASE_CSV.is_file(), f'Falta el catálogo de etapas previas: {BASE_CSV}'
print('Proyecto:', ROOT)
print('Catálogo anterior:', BASE_CSV)


Proyecto: C:\Users\jvera\Documents\memoria_chilena
Catálogo anterior: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa7\catalogo_para_busqueda.csv


## 1. Minisitios y políticas de acceso

Piloto limitado a tres áreas diferentes, para medir generalización fuera de religiosidad andina.

In [3]:
# Son minisitios verificados, distintos del piloto sobre religiosidad andina.
MINISITIOS = [
    ('Sufragio femenino (1865–1952)', 'https://www.memoriachilena.gob.cl/602/w3-article-686839.html'),
    ('Memorias de formación de la nación (1810–1879)', 'https://www.memoriachilena.gob.cl/602/w3-article-100701.html'),
    ('Fiestas del centenario (1910)', 'https://www.memoriachilena.gob.cl/602/w3-article-710.html'),
]
MAX_DOCUMENTOS_POR_MINISITIO = 8  # piloto acotado; no descarga todo el portal
PAUSA_SEGUNDOS = 2.5            # robots.txt declara Crawl-Delay: 2
LIMITE_BYTES = 6_000_000         # solo HTML y RDF; no PDF
SITIO = 'www.memoriachilena.gob.cl'
AGENTE = 'MemoriaChilenaGraphRAG/0.4 (piloto-academico-metadatos)'
MC = Namespace('http://www.memoriachilena.cl/schemas/')
MCG = Namespace('https://example.org/mcgraphrag/etapa14/')


def fecha_utc():
    return datetime.now(timezone.utc).isoformat()


def texto_normalizado(s):
    return re.sub(r'\s+', ' ', str(s)).strip()


def canonica(url):
    p = urlparse(url)
    return p._replace(query='', fragment='').geturl()


def es_pagina(url):
    p = urlparse(url)
    return p.scheme=='https' and p.hostname==SITIO and bool(re.fullmatch(r'/602/w3-article-\d+\.html',p.path))


def articulo(url):
    m=re.search(r'w3-article-(\d+)\.html$',urlparse(url).path)
    if not m: raise ValueError('No es una ficha de Memoria Chilena: '+url)
    return m.group(1)


def soup_html(content):
    try:return BeautifulSoup(content.decode('utf-8-sig'),'html.parser')
    except UnicodeDecodeError:return BeautifulSoup(content,'html.parser')


def own_rdf_link(soup, pagina):
    identificador = articulo(pagina)
    for a in soup.select('a[href]'):
        link = canonica(urljoin(pagina,a.get('href','').strip()))
        p=urlparse(link)
        if (p.scheme=='https' and p.hostname==SITIO
            and p.path.endswith('/rdf-article-'+identificador+'.rdf')):
            return link
    return ''


def documentos_de_minisitio(soup, pagina):
    # Aceptar solamente la sección editorial «Documentos».
    # No confundir enlaces de navegación, temas relacionados ni bibliografía.
    hs = soup.find_all(re.compile(r'^h[1-6]$'))
    def normhead(value):
        return ''.join(c for c in unicodedata.normalize('NFKD',texto_normalizado(value).casefold()) if not unicodedata.combining(c))
    inicio = next((h for h in hs if normhead(h.get_text(' ',strip=True)) == 'documentos'), None)
    if inicio is None: return []
    detener = {'imagenes','audiovisual','cronologia','bibliografia','links','temas relacionados','bnd'}
    resultado, vistos = [], set()
    for item in inicio.next_elements:
        if not isinstance(item,Tag): continue
        if re.fullmatch(r'h[1-6]',item.name or '') and item is not inicio:
            if normhead(item.get_text(' ',strip=True)) in detener: break
        if item.name!='a' or not item.get('href'):continue
        url=canonica(urljoin(pagina,item['href']))
        etiqueta=texto_normalizado(item.get_text(' ',strip=True))
        if not es_pagina(url) or url==pagina or url in vistos:continue
        if not etiqueta or etiqueta.casefold() in ('descargar','rdf','imprimir','compartir'):continue
        resultado.append({'documento_url':url,'titulo_enlace':etiqueta,'minisitio_url':pagina,
                          'tipo_enlace':'aparece_en_seccion_documentos_html'})
        vistos.add(url)
    return resultado


def datos_ficha_html(soup):
    h=soup.find('h1')
    titulo=texto_normalizado(h.get_text(' ',strip=True)) if h else ''
    cadenas=[texto_normalizado(x) for x in soup.stripped_strings]
    derechos=''
    for i,x in enumerate(cadenas):
        if x.casefold().startswith('propiedad intelectual:'):
            derechos=x.split(':',1)[1].strip() or (cadenas[i+1] if i+1<len(cadenas) else '')
            break
    return titulo, derechos


def metadata_rdf(g,pagina):
    # Solo el sujeto con URL exacta de la ficha. No contar los URI secundarios como obras.
    sujeto=URIRef(pagina)
    def valores(pred):
        return [texto_normalizado(o) for o in g.objects(sujeto,pred)
                if isinstance(o,Literal) and texto_normalizado(o)]
    def primero(pred):
        v=valores(pred)
        return v[0] if v else ''
    materias=set()
    unidades=set()
    for obj in g.objects(sujeto,DC.subject):
        for v in g.objects(obj,MC.tema):
            if isinstance(v,Literal):materias.add(texto_normalizado(v))
    for obj in g.objects(sujeto,DC.relation):
        for v in g.objects(obj,MC.ut):
            if isinstance(v,Literal):unidades.add(texto_normalizado(v))
    return {'titulo_rdf':primero(DC.title),'autor':primero(DC.creator),
            'fecha':primero(DCTERMS.temporal) or primero(DC.date),
            'mcid':primero(MC.mcid),
            'materias':' | '.join(sorted(materias)),
            'unidades':' | '.join(sorted(unidades)),
            'notas':' | '.join(valores(DC.description)[:3]),
            'rdf_sujeto_existe': any(g.triples((sujeto,None,None)))}


def csv_guardar(path, filas, campos):
    with Path(path).open('w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=campos,extrasaction='ignore')
        writer.writeheader()
        writer.writerows(filas)


class Fetcher:
    def __init__(self,pause=PAUSA_SEGUNDOS):
        self.sesion=requests.Session()
        self.sesion.headers.update({'User-Agent':AGENTE,'Accept':'text/html,application/rdf+xml;q=0.9,*/*;q=0.3'})
        self.last=0.0
        self.pause=max(2.0,float(pause))
        self.robots=RobotFileParser()
        u='https://'+SITIO+'/robots.txt'
        resp=self.sesion.get(u,timeout=25,allow_redirects=False)
        resp.raise_for_status()
        self.robots.parse(resp.text.splitlines())
        self.last=time.monotonic()
        delay=self.robots.crawl_delay('*')
        if delay is not None:self.pause=max(self.pause,float(delay))

    def fetch(self,url,redirecciones=0):
        if redirecciones>3:raise RuntimeError('Demasiadas redirecciones')
        p=urlparse(url)
        if p.scheme!='https' or p.hostname!=SITIO:raise PermissionError('URL fuera del dominio autorizado')
        if not self.robots.can_fetch(AGENTE,url):raise PermissionError('Prohibido por robots.txt')
        espera=self.pause-(time.monotonic()-self.last)
        if espera>0:time.sleep(espera)
        response=self.sesion.get(url,stream=True,timeout=40,allow_redirects=False)
        self.last=time.monotonic()
        try:
            if 300<=response.status_code<400:
                next_url=urljoin(url,response.headers.get('Location',''))
                return self.fetch(next_url,redirecciones+1)
            response.raise_for_status()
            pedazos,total=[],0
            for bloque in response.iter_content(64*1024):
                if not bloque:continue
                total+=len(bloque)
                if total>LIMITE_BYTES:raise ValueError('Archivo HTML/RDF supera 6MB')
                pedazos.append(bloque)
            return b''.join(pedazos)
        finally:response.close()


def cached_fetch(fetcher, url, extension):
    c = CACHE/(articulo(url) if es_pagina(url) else re.search(r'rdf-article-(\d+)',url).group(1))
    c = c.with_suffix(extension)
    if c.is_file():
        return c.read_bytes(),'cache'
    datos=fetcher.fetch(url)
    c.write_bytes(datos)
    return datos,'red'

print('Configuración lista: 3 minisitios, máximo', MAX_DOCUMENTOS_POR_MINISITIO, 'documentos por minisitio.')


Configuración lista: 3 minisitios, máximo 8 documentos por minisitio.


## 2. Extracción, deduplicación y grafo curatorial

Se conservan los diez registros anteriores; los documentos adicionales se identifican por **URL de ficha**. No convertimos enlaces en relaciones históricas.

In [4]:
def ejecutar_etapa14(fetcher=None, destino=None):
    carpeta=Path(destino) if destino else OUT/'etapa14'
    carpeta.mkdir(parents=True,exist_ok=True)
    rdf_dir=carpeta/'rdf_nuevo'
    rdf_dir.mkdir(parents=True,exist_ok=True)
    # Evitar RDF huérfanos al ejecutar de nuevo; se conserva la caché de descargas.
    for stale in rdf_dir.glob('*.rdf'):
        stale.unlink()
    if fetcher is None:fetcher=Fetcher()

    base=[]
    with BASE_CSV.open(encoding='utf-8-sig',newline='') as f:
        base=list(csv.DictReader(f))
    assert base, 'El catálogo original está vacío'
    catalogo={}
    for row in base:
        url=canonica(row['ficha'])
        catalogo[url]={**row,'ficha':url,'id_interno':row.get('mcid') or 'ART-'+articulo(url),
                       'origen':'etapa7','estado_html':'heredado','estado_rdf':'heredado',
                       'fecha_consulta_utc':'','hash_rdf_sha256':'','minisitios':
                       'https://www.memoriachilena.gob.cl/602/w3-article-790.html'}
    membresias=[]
    for row in base:
        membresias.append({'documento_url':row['ficha'],
                           'minisitio_url':'https://www.memoriachilena.gob.cl/602/w3-article-790.html',
                           'titulo_enlace':row.get('titulo',''),
                           'tipo_enlace':'heredado_del_catalogo_de_etapa7'})
    errors=[]; seeds=[]; rastreo=[]; raw=Graph()
    # Identificar primero los documentos de cada sección, sin seguir enlaces ajenos a ella.
    docs={}
    for nombre,url in MINISITIOS:
        info={'minisitio':nombre,'url':url,'estado':'','titulo_html':'','documentos_encontrados':0,
              'documentos_seleccionados':0,'rdf_url':'','rdf_estado':''}
        try:
            html,via=cached_fetch(fetcher,url,'.html')
            soup=soup_html(html)
            info['titulo_html']=datos_ficha_html(soup)[0]
            links=documentos_de_minisitio(soup,url)
            info['documentos_encontrados']=len(links)
            selecion=links[:MAX_DOCUMENTOS_POR_MINISITIO]
            info['documentos_seleccionados']=len(selecion)
            info['estado']='ok' if links else 'sin_documentos_identificados'
            for e in selecion:
                membresias.append(e)
                docs[e['documento_url']]=docs.get(e['documento_url'],e['titulo_enlace'])
            own=own_rdf_link(soup,url)
            info['rdf_url']=own
            if own:
                try:
                    datos,via_rdf=cached_fetch(fetcher,own,'.rdf')
                    if len(datos)<LIMITE_BYTES:
                        g=Graph();g.parse(data=datos,format='xml',publicID=own)
                        raw+=g
                        (rdf_dir/('minisitio-'+articulo(url)+'.rdf')).write_bytes(datos)
                        info['rdf_estado']='ok'
                        rastreo.append({'url':own,'pagina_origen':url,'tipo':'rdf_minisitio',
                            'bytes':len(datos),'sha256':hashlib.sha256(datos).hexdigest(),
                            'triples_rdf':len(g),'via':via_rdf})
                except Exception as ex:
                    info['rdf_estado']='error'
                    errors.append({'fase':'rdf_minisitio','url':own,'detalle':type(ex).__name__+': '+str(ex)[:300]})
            else:info['rdf_estado']='sin_enlace'
        except Exception as ex:
            info['estado']='error'
            errors.append({'fase':'html_minisitio','url':url,'detalle':type(ex).__name__+': '+str(ex)[:300]})
        seeds.append(info)
        print('Minisitio:',nombre,'→',info['documentos_seleccionados'],'fichas;',info['estado'])

    for pos,(url,titulo_enlace) in enumerate(sorted(docs.items()),1):
        anterior=catalogo.get(url)
        fila={**(anterior or {}),'ficha':url,
              'titulo':anterior.get('titulo','') if anterior else titulo_enlace,
              'id_interno':(anterior or {}).get('id_interno') or 'ART-'+articulo(url),
              'origen':(anterior or {}).get('origen','etapa14'),
              'estado_html':'no_intentado','estado_rdf':'no_intentado',
              'fecha_consulta_utc':fecha_utc(),'hash_rdf_sha256':'',
              'rdf_fuente':(anterior or {}).get('rdf_fuente',''),
              'derechos':(anterior or {}).get('derechos',''),
              'mcid':(anterior or {}).get('mcid',''),
              'autor':(anterior or {}).get('autor',''),
              'fecha':(anterior or {}).get('fecha',''),
              'materias':(anterior or {}).get('materias',''),
              'unidades':(anterior or {}).get('unidades',''),
              'notas':(anterior or {}).get('notas',''),
              'datos':(anterior or {}).get('datos','')}
        try:
            html,via=cached_fetch(fetcher,url,'.html')
            soup=soup_html(html)
            titulo_html,derechos=datos_ficha_html(soup)
            if titulo_html and titulo_html.casefold()!='memoria chilena':fila['titulo']=titulo_html
            if derechos:fila['derechos']=derechos
            fila['estado_html']='ok'
            rdf=own_rdf_link(soup,url)
            if rdf:
                fila['rdf_fuente']=rdf
                try:
                    datos,via_rdf=cached_fetch(fetcher,rdf,'.rdf')
                    g=Graph();g.parse(data=datos,format='xml',publicID=rdf)
                    raw+=g
                    fila['estado_rdf']='ok'
                    fila['hash_rdf_sha256']=hashlib.sha256(datos).hexdigest()
                    d=metadata_rdf(g,url)
                    if not d.pop('rdf_sujeto_existe'):
                        fila['estado_rdf']='sin_sujeto_principal'
                    for campo in ('autor','fecha','mcid','materias','unidades','notas'):
                        if d.get(campo):fila[campo]=d[campo]
                    if d.get('titulo_rdf'):fila['titulo']=d['titulo_rdf']
                    if fila.get('mcid'):fila['id_interno']=fila['mcid']
                    (rdf_dir/('article-'+articulo(url)+'.rdf')).write_bytes(datos)
                    rastreo.append({'url':rdf,'pagina_origen':url,'tipo':'rdf_documento',
                        'bytes':len(datos),'sha256':fila['hash_rdf_sha256'],
                        'triples_rdf':len(g),'via':via_rdf})
                except Exception as ex:
                    fila['estado_rdf']='error'
                    errors.append({'fase':'rdf_documento','url':rdf,'detalle':type(ex).__name__+': '+str(ex)[:300]})
            else:
                fila['estado_rdf']='sin_enlace_rdf'
        except Exception as ex:
            fila['estado_html']='error'
            errors.append({'fase':'html_documento','url':url,'detalle':type(ex).__name__+': '+str(ex)[:300]})
        catalogo[url]=fila
        print(f'  Ficha {pos}/{len(docs)}:',fila['titulo'][:65], '| RDF:',fila['estado_rdf'])

    por_doc={}
    for m in membresias:
        por_doc.setdefault(m['documento_url'],set()).add(m['minisitio_url'])
    for url,fila in catalogo.items():
        fila['minisitios']=' | '.join(sorted(por_doc.get(url,set())))

    # Grafo CURATORIAL: relaciones de catalogación y procedencia, no hechos históricos.
    grafo=Graph()
    grafo.bind('mcg',MCG);grafo.bind('dc',DC);grafo.bind('dct',DCTERMS);grafo.bind('prov',PROV)
    for name,url in [('Religiosidad andina','https://www.memoriachilena.gob.cl/602/w3-article-790.html')]+MINISITIOS:
        sitio=URIRef(url)
        grafo.add((sitio,RDF.type,MCG.MinisitioCatalografico))
        grafo.add((sitio,DC.title,Literal(name)))
    for url,d in catalogo.items():
        s=URIRef(url)
        grafo.add((s,RDF.type,MCG.FichaDocumental))
        for clave,prop in [('titulo',DC.title),('autor',DC.creator),('fecha',MCG.fechaCatalogo),
                           ('mcid',MCG.identificadorCatalografico),('derechos',MCG.derechosDeclarados),
                           ('materias',MCG.materiasCatalograficas),('unidades',MCG.unidadesCatalograficas)]:
            if d.get(clave):grafo.add((s,prop,Literal(str(d[clave]))))
        grafo.add((s,PROV.wasDerivedFrom,URIRef(url)))
        if d.get('rdf_fuente'):
            grafo.add((s,MCG.rdfCatalografico,URIRef(d['rdf_fuente'])))
    for m in membresias:
        grafo.add((URIRef(m['documento_url']),DCTERMS.isPartOf,URIRef(m['minisitio_url'])))
    fields=['mcid','id_interno','ficha','titulo','autor','fecha','materias','unidades',
            'notas','datos','derechos','rdf_fuente','minisitios','origen','estado_html','estado_rdf',
            'fecha_consulta_utc','hash_rdf_sha256']
    lista=sorted(catalogo.values(),key=lambda x:x['ficha'])
    csv_guardar(carpeta/'catalogo_ampliado.csv',lista,fields)
    csv_guardar(carpeta/'membresias_minisitios.csv',membresias,
               ['documento_url','minisitio_url','titulo_enlace','tipo_enlace'])
    csv_guardar(carpeta/'minisitios_auditados.csv',seeds,
               ['minisitio','url','estado','titulo_html','documentos_encontrados',
                'documentos_seleccionados','rdf_url','rdf_estado'])
    csv_guardar(carpeta/'rdf_fuentes.csv',rastreo,
               ['url','pagina_origen','tipo','bytes','sha256','triples_rdf','via'])
    csv_guardar(carpeta/'errores_etapa14.csv',errors,['fase','url','detalle'])
    grafo.serialize(destination=str(carpeta/'grafo_curatorial_etapa14.ttl'),format='turtle')
    raw.serialize(destination=str(carpeta/'rdf_original_nuevo.ttl'),format='turtle')
    resumen={'etapa':14,'generado_utc':fecha_utc(),
            'fichas_base':len(base),'fichas_finales':len(catalogo),
            'fichas_nuevas':sum(u not in {x['ficha'] for x in base} for u in catalogo),
            'minisitios_solicitados':len(MINISITIOS),
            'minisitios_procesados':sum(s['estado']=='ok' for s in seeds),
            'enlaces_curatoriales_registrados':len(membresias),
            'fichas_pendientes_nuevas':sum(x.get('origen')=='etapa14' and x.get('estado_rdf')!='ok' for x in lista),
            'rdf_fuentes_descargadas':len(rastreo),
            'triples_originales_nuevos':len(raw),
            'triples_grafo_curatorial':len(grafo),
            'errores':len(errors),
            'modelo_llm':'no usado',
            'pdf_y_ocr':'no descargados ni procesados',
            'reindexar_embeddings':True,
            'advertencias':['Las conexiones solo prueban co-presencia en minisitios, NO hechos históricos.',
              'Los 384-d embeddings de etapa 10 corresponden a las diez fichas antiguas: no sirven para el catálogo ampliado.',
              'El campo fecha es catalográfico: no asumir que equivale al año de edición.',
              'Los metadatos extraídos pueden incluir errores o vacíos; mantener URL de procedencia.',
              'No realizar evaluación con métricas previas sin nuevos juicios de relevancia.']}
    (carpeta/'informe_etapa14.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
    (carpeta/'SIGUIENTE_PASO.md').write_text(
        '# Etapa 15: reindexación y evaluación a mayor escala\n\n'
        '1. Revisar el catálogo ampliado y los errores.\n'
        '2. Generar TF-IDF + MiniLM sobre TODAS las fichas, no reutilizar embeddings de etapa 10.\n'
        '3. Evaluar la recuperación léxica / semántica / RRF / RRF+grafo con nuevas consultas y juicios.\n'
        '4. Conectar Qwen (Groq) solo después de validar evidencias y citas.\n',encoding='utf-8')
    return resumen,carpeta


## 3. Ejecutar y generar ZIP automáticamente

In [5]:
# Ejecuta la ampliación en vivo. Los HTML/RDF se guardan en caché por si vuelves a ejecutar.
RESUMEN, CARPETA = ejecutar_etapa14()
ARCHIVO = OUT/'etapa14.zip'
with zipfile.ZipFile(ARCHIVO,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CARPETA.rglob('*')):
        if p.is_file(): z.write(p,arcname=p.relative_to(CARPETA))
assert zipfile.is_zipfile(ARCHIVO)
print(json.dumps(RESUMEN,ensure_ascii=False,indent=2))
print('\nArchivo final:',ARCHIVO)
display(FileLink(str(ARCHIVO)))
if RESUMEN['fichas_nuevas']==0:
    print('ATENCIÓN: no se incorporaron fichas nuevas. Revisa minisitios_auditados.csv y errores_etapa14.csv.')
else:
    print('Etapa 14 completada. Conserva las etapas anteriores y compárteme etapa14.zip.')


Minisitio: Sufragio femenino (1865–1952) → 8 fichas; ok
Minisitio: Memorias de formación de la nación (1810–1879) → 8 fichas; ok
Minisitio: Fiestas del centenario (1910) → 8 fichas; ok
  Ficha 1/24: El voto femenino | RDF: ok
  Ficha 2/24: Colección de discursos de don Abdón Cifuentes. Tomo primero | RDF: ok
  Ficha 3/24: El sufrajio femenino | RDF: ok
  Ficha 4/24: Sufragio femenino | RDF: ok
  Ficha 5/24: Ley general sobre inscripciones electorales | RDF: ok
  Ficha 6/24: Ley de Elecciones (Texto definitivo) | RDF: ok
  Ficha 7/24: Ley general de elecciones No. 9334 de marzo de 1949 | RDF: ok
  Ficha 8/24: El chileno consolado en los presidios, o, filosofía de la religió | RDF: ok
  Ficha 9/24: Fundación del Instituto Nacional: (1810-1813) | RDF: ok
  Ficha 10/24: Colección de los artículos de Jotabeche publicados en el Mercurio | RDF: ok
  Ficha 11/24: Los festejos del centenario de la independencia : Chile en 1910 | RDF: ok
  Ficha 12/24: Nota de uno de los diputados de Rancagua al

C:\Users\jvera\Documents\memoria_chilena\data\output\etapa14.zip

Etapa 14 completada. Conserva las etapas anteriores y compárteme etapa14.zip.


## 4. Revisar la ampliación

Si faltan documentos, consulta `errores_etapa14.csv`. Los embeddings anteriores deben reindexarse en la etapa 15.

In [6]:
# Una consulta rápida sobre el nuevo catálogo; no usa LLM ni descarga nada.
import pandas as pd
CATALOGO_14=pd.read_csv(CARPETA/'catalogo_ampliado.csv',dtype=str,keep_default_na=False)
print('Fichas:',len(CATALOGO_14),'| nuevas:',int((CATALOGO_14.origen=='etapa14').sum()))
display(CATALOGO_14[['id_interno','titulo','autor','fecha','minisitios','estado_rdf']].tail(15))


Fichas: 34 | nuevas: 24


,id_interno,titulo,autor,fecha,minisitios,estado_rdf
19,MC0049519,"Zig-Zag : año VI, números 280-292, 2 de julio ...",,start=1910;end=1910,https://www.memoriachilena.gob.cl/602/w3-artic...,ok
20,MC0013341,Cuando Chile cumplió 100 años,"Calderón, Alfonso, 1930-2009",start=1973;end=1973,https://www.memoriachilena.gob.cl/602/w3-artic...,ok
21,MC0014753,Doctrina christiana y catecismo para la instru...,,1584,https://www.memoriachilena.gob.cl/602/w3-artic...,heredado
22,MC0014754,Parte primera de la Chrónica del Perú : que tr...,"Cieza de León, Pedro de, ca. 1520-1554",1553,https://www.memoriachilena.gob.cl/602/w3-artic...,heredado
23,MC0014755,"Primera parte de los commentarios reales, que ...","Vega, Garcilaso de la",start=1609;end=1609,https://www.memoriachilena.gob.cl/602/w3-artic...,heredado
24,MC0014757,Historia natural y moral de las Indias : en qu...,"Acosta, José de",start=1590;end=1590,https://www.memoriachilena.gob.cl/602/w3-artic...,heredado
25,MC0014760,Extirpación de la idolatria del Piru,"Arriaga, Pablo José de",start=1621;end=1621,https://www.memoriachilena.gob.cl/602/w3-artic...,heredado
26,MC0014761,Vocabulario de la lengua aymara,"Bertonio, Ludovico",start=1612;end=1612,https://www.memoriachilena.gob.cl/602/w3-artic...,heredado
27,MC0016815,"Nuestra inferioridad económica : sus causas, s...","Encina, Francisco Antonio",start=1955;end=1981,https://www.memoriachilena.gob.cl/602/w3-artic...,ok
28,MC0018305,Actas del Cabildo : año de 1810,,start=1541;end=1810,https://www.memoriachilena.gob.cl/602/w3-artic...,ok
